# Exercises XP Ninja ? Intelligent Document Assistant

XP Ninja: Build an Intelligent Document Assistant

## What you'll learn
- Combine foundational and advanced prompt engineering techniques.
- Apply CoT, role prompting, and memory chaining in a real scenario.
- Design a dynamic, multi-step LLM workflow that adapts to user input.
- Mitigate limitations like hallucination and bias.

## What you'll build
- An LLM-powered Document Assistant that: summarizes the contract, answers follow-up questions, keeps context, and self-critiques.

### Document (input)
```
Service Agreement ? Excerpt

This Service Agreement ("Agreement") is made effective as of March 1, 2025, by and between BrightLine Technologies Ltd. ("Provider") and NovaWare Systems Inc. ("Client").

Scope of Work: Provider shall deliver cloud infrastructure management services, including monitoring, incident response, and monthly reporting, as described in Exhibit A.

Payment Terms: Client agrees to pay a fixed monthly fee of $12,000, payable within 30 days of receipt of invoice. Late payments will incur a 2% penalty per month.

Term and Termination: This Agreement shall commence on March 1, 2025, and remain in effect for 12 months. Either party may terminate with 30 days? written notice.

Confidentiality: Both parties agree to protect the confidentiality of proprietary or sensitive information shared during the course of the engagement.

Limitation of Liability: Provider?s total liability shall not exceed the fees paid by Client in the 3 months prior to a claim. Provider is not liable for indirect or consequential damages.

Governing Law: This Agreement shall be governed by the laws of the State of California.
```

## Helper: Run a prompt
Uses `ollama run` if available (default model: `llama3`, override with `OLLAMA_MODEL`). If Ollama is missing or fails, prints the prompt in dry-run mode instead of crashing.

In [1]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 200):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None

## Step 1: Initial Summary Prompt
Write a few-shot or Chain-of-Thought prompt to summarize the document in plain English. Include: responsibilities, payment terms, termination, liability.

In [2]:
# TODO: craft a few-shot or CoT summary prompt
summary_prompt = '''
You are a legal document assistant.

Task:
Read the contract excerpt below and generate a clear, concise summary in plain English.

Instructions:
- Analyze the contract internally before answering.
- Do not explain your reasoning.
- Produce only the final summary.
- Include the following:
  1. The main responsibilities of the Provider and the Client.
  2. The payment terms.
  3. The termination clause.
  4. The limitation of liability.
- Keep the summary between 120 and 180 words.

Contract:

Service Agreement – Excerpt

This Service Agreement ("Agreement") becomes effective on March 1, 2025, between BrightLine Technologies Ltd. ("Provider") and NovaWare Systems Inc. ("Client").

Scope of Work:
The Provider shall deliver cloud infrastructure management services, including monitoring, incident response, and monthly reporting, as described in Appendix A.

Payment Terms:
The Client agrees to pay a monthly fee of $12,000, due within 30 days of invoice receipt. Late payments incur a 2% monthly penalty.

Term and Termination:
The Agreement begins on March 1, 2025, and continues for 12 months. Either party may terminate the Agreement with 30 days' written notice.

Confidentiality:
Both parties agree to protect confidential and sensitive information shared during the engagement.

Limitation of Liability:
The Provider's total liability shall not exceed the fees paid by the Client during the three months preceding the claim. The Provider is not liable for indirect or consequential damages.

Governing Law:
This Agreement is governed by the laws of the State of California.
'''
summary_prompt

'\nYou are a legal document assistant.\n\nTask:\nRead the contract excerpt below and generate a clear, concise summary in plain English.\n\nInstructions:\n- Analyze the contract internally before answering.\n- Do not explain your reasoning.\n- Produce only the final summary.\n- Include the following:\n  1. The main responsibilities of the Provider and the Client.\n  2. The payment terms.\n  3. The termination clause.\n  4. The limitation of liability.\n- Keep the summary between 120 and 180 words.\n\nContract:\n\nService Agreement – Excerpt\n\nThis Service Agreement ("Agreement") becomes effective on March 1, 2025, between BrightLine Technologies Ltd. ("Provider") and NovaWare Systems Inc. ("Client").\n\nScope of Work:\nThe Provider shall deliver cloud infrastructure management services, including monitoring, incident response, and monthly reporting, as described in Appendix A.\n\nPayment Terms:\nThe Client agrees to pay a monthly fee of $12,000, due within 30 days of invoice receipt. 

In [3]:
# Optional: test
run_prompt(summary_prompt)

[dry-run] ollama not installed. Prompt to send: 
You are a legal document assistant.

Task:
Read the contract excerpt below and generate a clear, concise summary in plain English.

Instructions:
- Analyze the contract internally before answering.
- Do not explain your reasoning.
- Produce only the final summary.
- Include the following:
  1. The main responsibilities of the Provider and the Client.
  2. The payment terms.
  3. The termination clause.
  4. The limitation of liability.
- Keep the summary between 120 and 180 words.

Contract:

Service Agreement – Excerpt

This Service Agreement ("Agreement") becomes effective on March 1, 2025, between BrightLine Technologies Ltd. ("Provider") and NovaWare Systems Inc. ("Client").

Scope of Work:
The Provider shall deliver cloud infrastructure management services, including monitoring, incident response, and monthly reporting, as described in Appendix A.

Payment Terms:
The Client agrees to pay a monthly fee of $12,000, due within 30 days 

## Step 2: Role-Based Follow-Up Q&A
Role: contract lawyer. Use Narrative-of-Thought or Instance-Adaptive CoT to answer user questions about the agreement.

In [4]:
# TODO: write the role-based Q&A prompt
qa_prompt = '''
You are an experienced contract lawyer.

Role:
You are advising a client about the service agreement below. Answer questions accurately using only the information contained in the agreement. Before answering, analyze the relevant clauses internally, but do not reveal your reasoning process. Provide only the final answer in clear, professional language.

Guidelines:
- Base every answer only on the contract.
- If the contract does not provide enough information, explicitly say so.
- Quote or refer to the relevant clause when appropriate.
- Explain legal terms in simple language.
- Avoid making assumptions or giving legal advice beyond what is stated in the agreement.

Contract:

Service Agreement – Excerpt

This Service Agreement ("Agreement") becomes effective on March 1, 2025, between BrightLine Technologies Ltd. ("Provider") and NovaWare Systems Inc. ("Client").

Scope of Work:
The Provider shall deliver cloud infrastructure management services, including monitoring, incident response, and monthly reporting, as described in Appendix A.

Payment Terms:
The Client agrees to pay a monthly fee of $12,000, due within 30 days of invoice receipt. Late payments incur a 2% monthly penalty.

Term and Termination:
The Agreement begins on March 1, 2025, and continues for 12 months. Either party may terminate the Agreement with 30 days' written notice.

Confidentiality:
Both parties agree to protect confidential and sensitive information shared during the engagement.

Limitation of Liability:
The Provider's total liability shall not exceed the fees paid by the Client during the three months preceding the claim. The Provider is not liable for indirect or consequential damages.

Governing Law:
This Agreement is governed by the laws of the State of California.

User Question:
Can you explain the limitation of liability clause?
'''
qa_prompt

'\nYou are an experienced contract lawyer.\n\nRole:\nYou are advising a client about the service agreement below. Answer questions accurately using only the information contained in the agreement. Before answering, analyze the relevant clauses internally, but do not reveal your reasoning process. Provide only the final answer in clear, professional language.\n\nGuidelines:\n- Base every answer only on the contract.\n- If the contract does not provide enough information, explicitly say so.\n- Quote or refer to the relevant clause when appropriate.\n- Explain legal terms in simple language.\n- Avoid making assumptions or giving legal advice beyond what is stated in the agreement.\n\nContract:\n\nService Agreement – Excerpt\n\nThis Service Agreement ("Agreement") becomes effective on March 1, 2025, between BrightLine Technologies Ltd. ("Provider") and NovaWare Systems Inc. ("Client").\n\nScope of Work:\nThe Provider shall deliver cloud infrastructure management services, including monitor

## Step 3: Memory Integration
Simulate context chaining so the assistant remembers the summary and answers accordingly. Choose a method: prior message passing, structured history, or describe vector store retrieval.

In [ ]:
# TODO: design the memory/context structure
memory_plan = '''
Memory Method: Structured Conversation History

1. Generate a summary of the contract.
2. Store the summary as persistent context.
3. Store each user question and assistant answer in chronological order.
4. For every new question:
   - Provide the contract summary.
   - Include the previous conversation history.
   - Append the new user question.
5. Instruct the assistant to answer using the summary and conversation history only.
6. If the answer cannot be found in the available context, state that the document does not provide that information.
'''
memory_plan

In [5]:
# TODO: sample prompt that injects summary + history
memory_prompt = '''
You are an experienced contract lawyer.

Use the conversation history and the contract summary below to answer the user's question.

Instructions:
- Reason internally but do not reveal your reasoning.
- Answer only from the supplied context.
- If the information is unavailable, clearly state that the agreement does not specify it.

=== Contract Summary ===

The agreement is between BrightLine Technologies Ltd. (Provider) and NovaWare Systems Inc. (Client). The Provider delivers cloud infrastructure management services including monitoring, incident response, and monthly reporting. The Client pays $12,000 per month within 30 days of invoice receipt, with a 2% monthly late fee. The agreement lasts 12 months and may be terminated by either party with 30 days' written notice. Both parties must protect confidential information. The Provider's liability is limited to the fees paid during the previous three months and excludes indirect or consequential damages. California law governs the agreement.

=== Conversation History ===

User: Can you explain the limitation of liability clause?
Assistant: The Provider's liability is capped at the fees paid during the three months preceding the claim and excludes indirect or consequential damages.

=== New User Question ===

Does the agreement specify what happens if the Provider misses a service deadline?
'''
memory_prompt

"\nYou are an experienced contract lawyer.\n\nUse the conversation history and the contract summary below to answer the user's question.\n\nInstructions:\n- Reason internally but do not reveal your reasoning.\n- Answer only from the supplied context.\n- If the information is unavailable, clearly state that the agreement does not specify it.\n\n=== Contract Summary ===\n\nThe agreement is between BrightLine Technologies Ltd. (Provider) and NovaWare Systems Inc. (Client). The Provider delivers cloud infrastructure management services including monitoring, incident response, and monthly reporting. The Client pays $12,000 per month within 30 days of invoice receipt, with a 2% monthly late fee. The agreement lasts 12 months and may be terminated by either party with 30 days' written notice. Both parties must protect confidential information. The Provider's liability is limited to the fees paid during the previous three months and excludes indirect or consequential damages. California law go

## Step 4: Mitigation & Refinement
Add a self-reflection or multi-agent critique step to review answers for accuracy and clarity.

In [ ]:
# TODO: write self-critique / refinement prompt
critique_prompt = '''
You are reviewing your previous answer as a contract law expert.

Task:
Evaluate your answer before presenting it to the user.

Review Checklist:
1. Is every statement supported by the contract?
2. Did you avoid making assumptions or providing legal advice beyond the document?
3. Is the explanation clear and easy for a non-lawyer to understand?
4. Did you answer every part of the user's question?
5. If you find any inaccuracies or unclear wording, rewrite the answer.

Do not reveal your reasoning process.
Output only the improved final answer.
'''
critique_prompt

In [6]:
# Optional: chain answer + critique
combined_prompt = '''
You are an experienced contract lawyer.

Step 1:
Read the contract and answer the user's question using only the information in the agreement.

Step 2:
Review your own answer using the following checklist:
- Is it factually correct?
- Is every statement supported by the contract?
- Are there unsupported assumptions?
- Is the explanation clear and concise?
- Could the wording be improved?

Step 3:
If necessary, revise the answer.

Do not reveal your reasoning or internal review.
Return only the final revised answer.

User Question:
Can you explain the limitation of liability clause?
'''
combined_prompt

"\nYou are an experienced contract lawyer.\n\nStep 1:\nRead the contract and answer the user's question using only the information in the agreement.\n\nStep 2:\nReview your own answer using the following checklist:\n- Is it factually correct?\n- Is every statement supported by the contract?\n- Are there unsupported assumptions?\n- Is the explanation clear and concise?\n- Could the wording be improved?\n\nStep 3:\nIf necessary, revise the answer.\n\nDo not reveal your reasoning or internal review.\nReturn only the final revised answer.\n\nUser Question:\nCan you explain the limitation of liability clause?\n"

In [7]:
run_prompt(combined_prompt)

[dry-run] ollama not installed. Prompt to send: 
You are an experienced contract lawyer.

Step 1:
Read the contract and answer the user's question using only the information in the agreement.

Step 2:
Review your own answer using the following checklist:
- Is it factually correct?
- Is every statement supported by the contract?
- Are there unsupported assumptions?
- Is the explanation clear and concise?
- Could the wording be improved?

Step 3:
If necessary, revise the answer.

Do not reveal your reasoning or internal review.
Return only the final revised answer.

User Question:
Can you explain the limitation of liability clause?

